---
<a id="simulation"></a>
## 21 — Simulation

| # | File | Difficulty | Key idea |
|---|------|------------|----------|
| 0059 | [score_tracker.ipynb](../0059.score_tracker.ipynb) | 🟢 | defaultdict + sorted tuple key |
| 0060 | [task_scheduler_sim.ipynb](../0060.task_scheduler_sim.ipynb) | 🟢 | Running clock + parse/split |
| 0064 | [min_operations_array.ipynb](../0064.min_operations_array.ipynb) | 🟢 | `prev+1−num` ops, `prev = num + ops` |
| 0071 | [num_recent_calls.ipynb](../0071.num_recent_calls.ipynb) | 🟢 | Deque — drop left while `t−q[0]>3000` |
| 0005 | [drone_delivery.ipynb](../0005.drone_delivery.ipynb) | 🟡 | Range=10, nearest station |
| 0120 | [encode_decode_strings.ipynb](../0120.encode_decode_strings.ipynb) | 🟡 | Length-prefix encode/decode |
| 0139 | [phone_battery_rotation.ipynb](../0139.phone_battery_rotation.ipynb) | 🟡 | Circular pointer, full/partial/gap |
| 0140 | [token_ring_cooldown.ipynb](../0140.token_ring_cooldown.ipynb) | 🟡 | Same — deque rotate variant |
| 0141 | [printer_slots_round_robin.ipynb](../0141.printer_slots_round_robin.ipynb) | 🟡 | Same circular template |
| 0142 | [circular_workers_with_rest.ipynb](../0142.circular_workers_with_rest.ipynb) | 🟡 | Same circular template |
| 0145 | [bridge_crossing_simulation.ipynb](../0145.bridge_crossing_simulation.ipynb) | 🔴 | 4 queues: left/right wait + left/right work |

# Multi Path Sort very important Seconday first then primary

In [1]:
from collections import defaultdict
   
def sortedScores(results):
    team = defaultdict(int)
    for itm in results:
        name, score = itm.split(' ')
        team[name] += int(score)
    teams = list(team.items()) 

#    teams.sort(key=lambda x: (-x[1], x[0]))
    # Multi-pass with list.sort() (stable)
    teams.sort(key=lambda x: x[0])                 # secondary: name asc
    teams.sort(key=lambda x: x[1], reverse=True)   # primary: score desc
    
    return [name + " " + str(score) for name, score in teams]

print(sortedScores(["Alice 10","Bob 20","Alice 15","Bob 5"]))
assert sortedScores(["Alice 10","Bob 20","Alice 15","Bob 5"])  == ["Alice 25","Bob 25"]       # given example 1 — tie → alpha
assert sortedScores(["Zara 100","Mike 50","Zara 10"])          == ["Zara 110","Mike 50"]       # given example 2
assert sortedScores(["Alice 5"])                               == ["Alice 5"]                 # single entry
assert sortedScores(["B 10","A 10","C 10"])                    == ["A 10","B 10","C 10"]      # all tied — alpha
assert sortedScores(["X 1","Y 100","Z 50"])                    == ["Y 100","Z 50","X 1"]      # descending score
assert sortedScores(["A 3","A 3","A 3"])                       == ["A 9"]                     # same player multi-round
assert sortedScores(["D 5","A 10","B 10","C 5"])               == ["A 10","B 10","C 5","D 5"] # mixed
print('All Pass!')

['Alice 25', 'Bob 25']
All Pass!


# Min Operations needed to enforce ascending

In [2]:
def minOperations(nums):
    nums2 = nums[:]
    if len(nums2) <= 1: return 0
    n = len(nums2)
    operations = 0
    for i in range(1,n):
        delta = max(0, nums2[i-1] - nums2[i] + 1)
        nums2[i] += delta
        operations += delta 
    return operations



print(minOperations([1,5,2,4,1]) )
assert minOperations([1,1,1])       == 3    # given example 1
assert minOperations([1,5,2,4,1])   == 14   # given example 2
assert minOperations([8])           == 0    # single element
assert minOperations([1,2,3])       == 0    # already strictly increasing
assert minOperations([3,2,1])       == 6    # descending
assert minOperations([0,0])         == 1    # two equal elements
assert minOperations([1,1,1,1])     == 6    # 0+1+2+3
assert minOperations([5,5,5,5,5])   == 10   # 0+1+2+3+4
print('All Pass!')

14
All Pass!


# Expiry Time Model LeetCode 933 

In [3]:

# no need for a clock cause time of arrival is the clock
# at tiem of arrival stamp exiry time in the deque
# at the new time (when somebody arrives)  .. eject everybody with stamp less than currentclock
# let the new one get in
from collections import deque

class RecentCounter:
    def __init__(self):
        self.q = deque()
       
    def ping(self, t):       
        self.q.append(3000+ t)
        while self.q and  self.q[0] < t :
            self.q.popleft()
        return len(self.q)
        

rc = RecentCounter()
print(rc.ping(1))
print(rc.ping(100) )
print(rc.ping(3001) )
print(rc.ping(3002) )

1
2
3
3


# Drone Delivery simulate disatance

In [4]:
def solution(target, stations):
    stations.sort()
    if not stations: return target
    walks = stations[0]
    pos = walks
    for d in stations:
        if pos >= target: #arrived
            break
        if pos > d:          # you are post the next drone .. you need to walk to next
            continue
        else:
            walks += d - pos
            pos = d + 10
    walks += max(0,  target - pos)                    # add more if not yet arrive .. only add positive values

    return walks


print( solution(27, [15, 7, 3, 10]) )
print( solution(21, [7, 4, 14]))           #SHould return 4
assert solution(21, [7, 4, 14])      == 4   # given example 1
assert solution(27, [15, 7, 3, 10])  == 7   # given example 2
assert solution(10, [])              == 10  # no stations — walk the whole way
assert solution(1,  [])              == 1   # minimal case, no stations
assert solution(5,  [3])             == 3   # one station, drone reaches target
assert solution(20, [10])            == 10  # walk to station, drone hits target exactly
assert solution(25, [10])            == 15  # walk to station, drone to 20, walk last 5
assert solution(16, [5, 12])         == 6   # walk to 5, drone to 15, walk last 1
print("All Done!!")

7
4
All Done!!


# 0120 Encode decode .. Treat string as a stream and use find to find delims

In [5]:
def encode(strs):
    out= ''
    for s in strs:
        n = len(s)
        encoded = str(n) + "~" + s
        out+=(encoded)
    return out


def decode(s):
    out = []
    i = 0 

    while i < len(s):
        sep = s.find("~", i)
        n = int(s[i:sep])
        start = sep + 1
        end = start + n
        out.append(s[start:end])
        i = end
    return out


def round_trip(strs):
    return decode(encode(strs))


decode(encode(["hello","world"]))
assert round_trip(["hello","world"])          == ["hello","world"]
assert round_trip(["a","b","c"])              == ["a","b","c"]
assert round_trip(["with spaces","and#spec"]) == ["with spaces","and#spec"]
assert round_trip([""])                       == [""]
assert round_trip(["hello"])                  == ["hello"]
print('All Pass!')

All Pass!


## 0139     Phone Battery Rotation (Custom Simulation)  cntr outside the loop and increment adter you use

In [7]:
# Count fully used batteries only
def solution(t, capacity, recharge):
    n = len(capacity)                 #i % n
    clock = 0
    ready_at  = [0] * n   # time when each battery is available again
    cntr = 0
    full_used = 0
    
    while clock < t:
        found = False
        for k in range(n):
            i = cntr %n
            cntr += 1
            if ready_at[i] <= clock:
                found = True
                remain = t - clock
                if capacity[i] <= remain:
                    full_used += 1
                    clock += capacity[i]
                    ready_at[i] = clock + recharge[i]
                    break   # <- key fix
                else:
                    return full_used
        if not found:
            return -1
    return full_used

print(solution(8,  [2,5],  [10,4]))
def test():
    assert solution(8,  [2,5],  [10,4])      == -1
    assert solution(10, [7,5],  [12,4])      == 1
    assert solution(6,  [3,3],  [1,1])       == 2
    assert solution(1,  [5],    [3])         == 0
    assert solution(5,  [5],    [3])         == 1
    print('All Pass!')   
test()

-1
All Pass!


# Used deque for looping

In [9]:
# Count fully used batteries only
from collections import deque 
def solution(t, capacity, recharge):
    n = len(capacity)                 #i % n
    clock = 0
    ready_at  = [0] * n   # time when each battery is available again
    order = deque(range(n))
    full_used = 0
    
    while clock < t:
        found = False
        for k in range(n):
            i = order[0]
            order.rotate(-1)
            if ready_at[i] <= clock:
                found = True
                remain = t - clock
                if capacity[i] <= remain:
                    full_used += 1
                    clock += capacity[i]
                    ready_at[i] = clock + recharge[i]
                    break   # <- key fix
                else:
                    return full_used
        if not found:
            return -1
    return full_used

print(solution(8,  [2,5],  [10,4]))
def test():
    assert solution(8,  [2,5],  [10,4])      == -1
    assert solution(10, [7,5],  [12,4])      == 1
    assert solution(6,  [3,3],  [1,1])       == 2
    assert solution(1,  [5],    [3])         == 0
    assert solution(5,  [5],    [3])         == 1
    print('All Pass!')   
test()

-1
All Pass!


# 0145 Bridge Crossing
## Bridge Crossing — 4-Queue Simulation

`n` workers, `k` packages on the **right** bank.  
Workers shuttle packages left.  
`time[i] = [leftToRight, pickOld, rightToLeft, putNew]`

**Rules:**
- Bridge fits **one person at a time**  
- **Right → Left has priority** over Left → Right (they carry packages)  
- Among same direction: **higher index = lower efficiency = crosses first**  
- Return the time the **last package is put down** on the left bank

**4 Queues:**

| Queue | Type | Contents |
|-------|------|----------|
| `left_wait` | max-heap (by index) | Ready to cross L→R |
| `right_wait` | max-heap (by index) | Ready to cross R→L |
| `left_work` | min-heap (by time) | Putting package down (putNew) |
| `right_work` | min-heap (by time) | Picking package up (pickOld) |

**Clock** = when the bridge becomes free next.  
At each tick: release finished workers → choose who crosses → advance clock.